In [ ]:
import random, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from victim.model_def import SimpleCNN

# This decides whether we use GPU or CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


In [5]:
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)

In [ ]:
# Create the model and move it to GPU/CPU
model = SimpleCNN().to(device)

In [ ]:
# Converts image to tensor and scales pixel values into [0,1]
transform = transforms.Compose([transforms.ToTensor()]) 

# Downloads MNIST into ./data the first time
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

print("train samples:", len(train_dataset))

100.0%
100.0%
100.0%
100.0%

train samples: 60000


In [ ]:
# Adam is a good default optimizer
optimizer = optim.Adam(model.parameters(), lr=1e-3) # 1e-3 = 0.001

# Standard loss for multi-class classification
loss_fn = nn.CrossEntropyLoss()

In [ ]:
model.train()

epochs = 10
for epoch in range(epochs):
    total_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)    
        optimizer.zero_grad() # Clears old gradients
        logits = model(images) # Forward pass gives raw scores 
        loss = loss_fn(logits, labels) # Loss measures how wrong those scores are  
        loss.backward() # Backprop computes gradients
        optimizer.step() # Update weights

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch {epoch+1}/{epochs} avg_loss={avg_loss:.4f}")

epoch 1/10 avg_loss=0.2948
epoch 2/10 avg_loss=0.0697
epoch 3/10 avg_loss=0.0506
epoch 4/10 avg_loss=0.0381
epoch 5/10 avg_loss=0.0304
epoch 6/10 avg_loss=0.0256
epoch 7/10 avg_loss=0.0218
epoch 8/10 avg_loss=0.0170
epoch 9/10 avg_loss=0.0149
epoch 10/10 avg_loss=0.0118


In [10]:
# Create test dataset 
test_dataset = datasets.MNIST(
    root="./data",
    train=False,  # IMPORTANT: this loads the 10k test images
    download=False,  # Already downloaded during training
    transform=transform
)

test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("Test samples:", len(test_dataset))

Test samples: 10000


In [11]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

final_acc = correct / total
print("Final Test Accuracy:", final_acc * 100)

Final Test Accuracy: 99.08


In [13]:
torch.save(model.state_dict(), "../victim/victim_model.pt")
print("saved victim_model.pt")

saved victim_model.pt
